# Conferência das sete análises de negócio

Fonte: exportação nativa da consulta DAX executada em 23/10/2026 no modelo Import Olist_Analytics_v2, aberto no Power BI Desktop. Compras de 04/09/2016 a 17/10/2018. Nenhum refresh foi acionado; a última atualização da origem não foi obtida.

Entradas: `00_controle.tsv`, `01_resultados.tsv` e a consulta exata `01_consultas_pendentes.dax`. Este notebook verifica a exportação local; não consulta novamente o Power BI.

Ambiente de autoria sem nbformat/Jupyter: as células foram executadas em ordem em um único namespace CPython e suas saídas foram registradas. A estrutura JSON foi conferida; o notebook não foi validado com nbformat nem inspecionado em Jupyter.

In [1]:
from pathlib import Path
import csv, json, math
base=Path.cwd()
if not (base/'01_resultados.tsv').exists():
    base=base/'analyses/powerbi_validacao'
with (base/'01_resultados.tsv').open(encoding='utf-8-sig',newline='') as f:
    raw=list(csv.DictReader(f,delimiter='\t'))
assert len(raw)==796
sections={}
for row in raw:
    a,g,m,v=[row[k] for k in ['[Analise]','[Grupo]','[Metrica]','[Valor]']]
    target=sections.setdefault(a,{}).setdefault(g,{})
    assert m not in target
    target[m]=float(v) if v else None
control=sections['00_Controle']['Total']
checks=[]
for sec,metrics in [('01_Mensal',['Pedidos','Entregues','ValorItens']),('02_Categorias',['ValorItens','Itens']),('04_UF',['Pedidos','Entregues','ValorItens','Frete'])]:
    for metric in metrics:
        total=sum((v[metric] or 0) for v in sections[sec].values())
        assert math.isclose(total,control[metric],rel_tol=0,abs_tol=.01),(sec,metric)
        checks.append((sec,metric,total))
log=sections['05_Logistica']['Total'];sat=sections['06_Satisfacao']['Total'];late=sections['07_AtrasoNotas']
assert math.isclose(sum((v['Participacao'] or 0) for v in sections['02_Categorias'].values()),1,abs_tol=1e-9)
assert log['NoPrazo']+log['Atrasados']==log['BasePontualidade']
assert math.isclose(log['TaxaAtraso'],log['Atrasados']/log['BasePontualidade'],abs_tol=1e-12)
assert sum(v['Pedidos'] for v in late.values())==log['BasePontualidade']
assert sat['Avaliacoes']==control['Avaliacoes']
assert sat['PedidosAvaliados']<=sat['Avaliacoes']
assert len(sections['03_VendedoresTop10'])==10
print('16 checagens aprovadas; 796 linhas; tolerância monetária de R$ 0,01 para ponto flutuante.')
for check in checks:print(check)


16 checagens aprovadas; 796 linhas; tolerância monetária de R$ 0,01 para ponto flutuante.
('01_Mensal', 'Pedidos', 99441.0)
('01_Mensal', 'Entregues', 96478.0)
('01_Mensal', 'ValorItens', 13591643.700000102)
('02_Categorias', 'ValorItens', 13591643.700000063)
('02_Categorias', 'Itens', 112650.0)
('04_UF', 'Pedidos', 99441.0)
('04_UF', 'Entregues', 96478.0)
('04_UF', 'ValorItens', 13591643.700001087)
('04_UF', 'Frete', 2251909.54)


In [2]:
for section, groups in sections.items():
    print(section, '-', len(groups), 'grupo(s)')
print('Taxa de atraso:', f"{log['TaxaAtraso']:.4%}")
print('Tempo médio de entrega:', round(log['DiasEntrega'], 4), 'dias')
for group in ['No prazo', 'Atrasado']:
    v=late[group]
    print(group, ':', int(v['PedidosAvaliados']), 'pedidos avaliados; média', round(v['NotaMediaPorPedido'], 4))
print('Diferença descritiva entre médias:', round(late['No prazo']['NotaMediaPorPedido']-late['Atrasado']['NotaMediaPorPedido'], 4))

00_Controle - 3 grupo(s)
01_Mensal - 25 grupo(s)
02_Categorias - 74 grupo(s)
03_VendedoresTop10 - 10 grupo(s)
04_UF - 27 grupo(s)
05_Logistica - 1 grupo(s)
06_Satisfacao - 1 grupo(s)
07_AtrasoNotas - 2 grupo(s)
Taxa de atraso: 6.7731%
Tempo médio de entrega: 12.5582 dias
No prazo : 89443 pedidos avaliados; média 4.2906
Atrasado : 6381 pedidos avaliados; média 2.2718
Diferença descritiva entre médias: 2.0188


In [3]:
cats=sorted(sections['02_Categorias'].items(), key=lambda x:x[1]['ValorItens'] or 0, reverse=True)
print('Líder em valor dos itens:', cats[0][0])
print('Participação das cinco maiores categorias:', f"{sum(v['Participacao'] for k,v in cats[:5]):.4%}")
print('Participação dos dez vendedores:', f"{sum(v['Participacao'] for v in sections['03_VendedoresTop10'].values()):.4%}")
print('Participação de SP no valor dos itens:', f"{sections['04_UF']['SP']['ValorItens']/control['ValorItens']:.4%}")
monthly=sections['01_Mensal']
peak=max(monthly, key=lambda m:monthly[m]['Pedidos'])
print('Pico mensal:', peak, int(monthly[peak]['Pedidos']), 'pedidos')
for year in ['2017','2018']:
    total=sum(v['ValorItens'] or 0 for m,v in monthly.items() if year+'-01'<=m<=year+'-08')
    print('Janeiro a agosto de', year, ': valor dos itens =', round(total, 2))

Líder em valor dos itens: beleza_saude
Participação das cinco maiores categorias: 39.7353%
Participação dos dez vendedores: 13.1496%
Participação de SP no valor dos itens: 38.2805%
Pico mensal: 2017-11 7544 pedidos
Janeiro a agosto de 2017 : valor dos itens = 3113000.32
Janeiro a agosto de 2018 : valor dos itens = 7385905.8


## Limitações e interpretação

- Todos os status nos recortes comerciais; entregues e elegibilidade logística são definidos na consulta.
- Tempo médio e pontualidade têm critérios distintos de inclusão.
- Avaliação por registro e média por pedido são métricas distintas.
- A diferença de notas entre atrasados e no prazo é associação observacional, sem controle de fatores ou conclusão causal.
- Não se somam pedidos distintos por categoria/vendedor. Valores monetários não definem receita líquida.
- Os anos de 2016 e 2018 são incompletos, e setembro/outubro de 2018 têm registros residuais.
- Ausências de resultado são mantidas como `None`; não imputadas como zero nas médias.
- As reconciliações históricas anteriores do documento não foram todas repetidas nesta execução.